### RAG Agent Chat Interface (Gradio)

Interactive notebook for chatting with a RAG agent using Gradio.

It uses **Gradio** to expose a web-based chat UI and **LangGraph** to orchestrate
the RAG workflow, including:
- loading application settings via `bootstrap`
- initializing the LLM and retriever with `build_rag`
- constructing the agent graph with `build_graph`
- handling conversational history and user queries
- collecting feedback through response voting (like / dislike)

### Import packages

In [10]:
import json
import sys
from datetime import datetime
from functools import partial
from pathlib import Path

import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

In [11]:
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [12]:
from src.bootstrap import bootstrap

### Load and Set Up LangGraph

All configuration is loaded through the `bootstrap` function.  
The resulting settings are passed to `build_rag`, which initializes RAG components.  
Finally, these components are used to construct the LangGraph.

In [ ]:
# Load all application context
app_context = await bootstrap()

# Build RAG components from settings
orchestrator = app_context.orchestrator

settings = app_context.settings

### Chatbot Response Voting

Chatbot responses support user voting.  
To enable persistent storage of vote data, logging must be enabled in the
application configuration.

Vote logging is activated by setting the following option in `settings.yaml`:

`logging - log_files - chat_bot_votes - enabled: true`

When enabled, all voted chatbot responses are automatically saved to the
configured `file_path`.

This mechanism allows vote data to be collected for later analysis.

In [14]:
LOG_CHAT_BOT_VOTES = None

# Enable vote logging if configured in settings
if settings["app"]["output"]["log_files"]["chat_bot_votes"]["enabled"]:
    LOG_CHAT_BOT_VOTES = settings["app"]["output"]["log_files"]["chat_bot_votes"]["file_path"]

In [15]:
def save_vote(record):
    # Resolve the path to the vote log file
    path = Path(LOG_CHAT_BOT_VOTES)

    # Ensure that the parent directory exists
    path.parent.mkdir(parents=True, exist_ok=True)

    # Append the vote record as a single JSON line
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

### Gradio & RAG History Adapter

To enable chat interaction, `gr.ChatInterface` is used.  
While it accepts multiple configuration parameters, the most important one is
the callback function (e.g. `answer_question`), which is responsible for handling
user input and returning model responses.

`gr.ChatInterface` requires the callback function to have the following
signature:

`fn(message: str, history: list[tuple[str, str]])`

The RAG agent, however, operates on a different conversation history
representation (`RAGState`). In this representation, the conversation is stored
as a list of dictionaries with explicit `role` and `content` fields.

Because of this mismatch in history formats, an adapter is required to transform
the Gradio-provided chat history into the format expected by the RAG agent.

The current RAG history message structure is defined as:

`{"role": "<user|assistant>", "content": "<text>"}`

This adapter ensures compatibility between the Gradio chat interface and the
internal RAG state representation without modifying either system’s internal
logic.


In [16]:
def gradio_to_rag_history(history):
    rag_history = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")

        if content is None or content == "":
            continue

        if role == "user":
            rag_history.append(HumanMessage(content=content))
        elif role == "assistant":
            rag_history.append(AIMessage(content=content))

    return rag_history

In [17]:
def gradio_to_rag_history(history):
    rag_history = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")

        if content is None or content == "":
            continue

        if role == "user":
            rag_history.append(HumanMessage(content=content))
        elif role == "assistant":
            rag_history.append(AIMessage(content=content))

    return rag_history


async def answer_question(message, history, orchestrator):
    messages = gradio_to_rag_history(history)

    state = {
        "question": message,
        "messages": messages,
        "documents": [],
        "answer": "",
        "user": {
            "user_id": "5",
            "is_professional": 1,
            "is_common": 1,
        },
    }

    result = await orchestrator.run(state)
    return result["answer"]

### Vote Handler

The `vote_handler` function processes user vote events submitted through the
Gradio chat interface. It captures metadata related to the voted chatbot
response and optionally persists this information to a log file.

In [18]:
def vote_handler(data: gr.LikeData, history):
    # Build a structured record for the vote event
    record = {
        "timestamp": datetime.now().isoformat(),
        "liked": data.liked,
        "assistant_response": data.value,
        "message_index": data.index,
        "history": history,
    }

    # Persist the vote record only if vote logging is enabled
    if LOG_CHAT_BOT_VOTES:
        print(record)
        save_vote(record)

### Main Callback

The `answer_question` function serves as the callback for `gr.ChatInterface`.
It receives the current user message and the accumulated Gradio chat history,
transforms the history into the format expected by the RAG agent, and invokes
the LangGraph pipeline.

**Responsibilities:**

* accept the user message and chat history from Gradio
* convert Gradio history into RAG-compatible history
* initialize the RAG state
* invoke the LangGraph graph
* return the generated answer to the chat interface

In [19]:
answer_fn = partial(answer_question, orchestrator=orchestrator)

### Gradio Chat Interface Setup

The chat interface is constructed using `gr.Blocks` to provide a flexible layout
and shared state. A `gr.Chatbot` component is used to render the conversation and
handle user interaction.

Vote handling is registered on the chatbot component to capture like and dislike
events for assistant responses. User messages are processed through a
`gr.ChatInterface`, which connects the chat UI to the application’s
`answer_question` callback.

In [ ]:
with gr.Blocks() as chat:
    # Chatbot component responsible for rendering the conversation
    chatbot = gr.Chatbot()

    # Register vote handler for like / dislike events on assistant messages
    chatbot.like(vote_handler, inputs=chatbot, outputs=None)

    # Chat interface wiring user input to the answer_question callback
    gr.ChatInterface(fn=answer_fn, chatbot=chatbot)

# Launch the gradio application
chat.launch()

### Additional information 

#### Important `gr.ChatInterface` Parameters

The following parameters are commonly used when integrating a RAG agent with
`gr.ChatInterface`:
- **`fn` — callback function; in this case, `answer_question`.**: callback function that receives the user message and chat history and returns
  the assistant response.

- **`chatbot`**: custom `gr.Chatbot` component used to control message rendering, height, or
  formatting.

- **`title`**: title displayed at the top of the chat interface.

- **`description`**: short description explaining the purpose of the chat interface.

- **`retry_btn` / `undo_btn` / `clear_btn`**:  controls for retrying the last message, undoing interactions, or clearing chat history.

- **`additional_inputs`**: extra UI components (e.g. dropdowns, sliders) passed to the callback function,
  often used for model or retrieval configuration.

#### Launch Configuration

The chat interface is typically started using the `launch()` method.
Common launch parameters include:

- **`debug=True`**: enables detailed error and logs traces in the UI, useful during development.

- **`share=True`**:Creates a temporary public URL for external access.
